In [ ]:
import pandas as pd
import numpy as np
import random
from datetime import datetime, timedelta

# Configuración para ver los dataframes completos
pd.set_option('display.max_columns', None)

In [ ]:
# 1. GENERAR PRODUCTOS (Catálogo de hardware)
productos_data = {
    'id_producto': range(1, 9),
    'nombre': [
        '  Procesador AMD Ryzen 7 7800X3D  ', # Espacios extra
        'Procesador AMD Ryzen 5 7600X',
        'Gabinete Mid-Tower Blanco (Sin RGB)',
        'teclado magnético irok mercury68', # Minúsculas
        'Monitor 240Hz 1ms 24"',
        'Mouse Ultraligero 50g',
        'Auriculares In-Ear KZ EDX Pro',
        'Memoria RAM 32GB DDR5 6000MHz'
    ],
    'categoria': ['Componentes', 'Componentes', 'Gabinetes', 'Periféricos', 'Monitores', 'Periféricos', 'Audio', 'Componentes'],
    'precio_unitario': [450.0, 230.0, 90.0, 110.0, 280.0, 75.0, 20.0, 120.0]
}
df_productos = pd.DataFrame(productos_data)

# Error intencional: Borramos el precio de un producto
df_productos.loc[2, 'precio_unitario'] = np.nan 

# 2. GENERAR CLIENTES
df_clientes = pd.DataFrame({
    'id_cliente': range(100, 150),
    'perfil_comprador': np.random.choice(['Gamer', 'Workstation', 'Casual'], size=50)
})

# 3. GENERAR VENTAS (Dataset principal)
np.random.seed(42)
num_ventas = 3000

df_ventas = pd.DataFrame({
    'id_venta': range(1000, 1000 + num_ventas),
    'fecha': [(datetime(2025, 1, 1) + timedelta(days=random.randint(0, 365), hours=random.randint(0,23))).strftime('%Y-%m-%d %H:%M:%S') for _ in range(num_ventas)],
    'id_cliente': [random.choice(df_clientes['id_cliente']) for _ in range(num_ventas)],
    'id_producto': [random.choice(df_productos['id_producto']) for _ in range(num_ventas)],
    # Errores en cantidad: números negativos y ceros
    'cantidad': np.random.choice([1, 2, 3, -1, 0], p=[0.75, 0.15, 0.03, 0.05, 0.02], size=num_ventas),
    # Errores en método de pago: valores nulos (None)
    'metodo_pago': np.random.choice(['Tarjeta', 'Transferencia', 'Crypto', None], p=[0.5, 0.3, 0.15, 0.05], size=num_ventas) 
})

# Error intencional: Duplicar filas
df_ventas = pd.concat([df_ventas, df_ventas.sample(30)]).sample(frac=1).reset_index(drop=True)

print("¡Datos generados con éxito!")

In [ ]:
# Vemos los errores en ventas (fijate los nulos y las filas totales)
print("--- INFO DE VENTAS ---")
df_ventas.info()

# Vemos un resumen estadístico (fijate que el mínimo en 'cantidad' es -1)
df_ventas.describe()

In [ ]:
# 1. PRODUCTOS: Limpiar textos y llenar nulos
df_productos['nombre'] = df_productos['nombre'].str.strip().str.title()
# Rellenamos el precio faltante del gabinete blanco (sabemos que era 90)
df_productos['precio_unitario'] = df_productos['precio_unitario'].fillna(90.0)

# 2. VENTAS: Eliminar duplicados
ventas_antes = len(df_ventas)
df_ventas = df_ventas.drop_duplicates()
print(f"Se eliminaron {ventas_antes - len(df_ventas)} filas duplicadas.")

# 3. VENTAS: Filtrar cantidades ilógicas (negativos y ceros)
df_ventas = df_ventas[df_ventas['cantidad'] > 0]

# 4. VENTAS: Rellenar métodos de pago vacíos
df_ventas['metodo_pago'] = df_ventas['metodo_pago'].fillna('No Especificado')

# 5. VENTAS: Convertir la fecha a formato datetime de Pandas
df_ventas['fecha'] = pd.to_datetime(df_ventas['fecha'])

print("¡Limpieza finalizada!")

In [ ]:
# Hacemos un JOIN (Merge) para unificar la información
df_master = df_ventas.merge(df_productos, on='id_producto', how='inner')
df_master = df_master.merge(df_clientes, on='id_cliente', how='inner')

# KPI CLAVE: Creamos la columna de ingreso total por venta
df_master['ingreso_total'] = df_master['cantidad'] * df_master['precio_unitario']

# Creamos columnas de mes y día para facilitar el análisis posterior
df_master['mes'] = df_master['fecha'].dt.month
df_master['dia_semana'] = df_master['fecha'].dt.day_name()

# Visualizamos el dataset final impecable
df_master.head()

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Configuración visual para que los gráficos se vean profesionales
sns.set_theme(style="whitegrid")
plt.figure(figsize=(15, 5))

# Gráfico 1: Ingresos totales por Categoría
plt.subplot(1, 2, 1)
ingresos_categoria = df_master.groupby('categoria')['ingreso_total'].sum().sort_values(ascending=False)
sns.barplot(x=ingresos_categoria.values, y=ingresos_categoria.index, palette="viridis")
plt.title('Ingresos Totales por Categoría', fontsize=14, fontweight='bold')
plt.xlabel('Ingreso en USD')
plt.ylabel('')

# Gráfico 2: Tendencia mensual de ventas (Cantidad de tickets)
plt.subplot(1, 2, 2)
ventas_mensuales = df_master.groupby('mes')['id_venta'].count()
sns.lineplot(x=ventas_mensuales.index, y=ventas_mensuales.values, marker='o', color='#2b7bba', linewidth=2.5)
plt.title('Tendencia de Ventas Mensuales (Tickets)', fontsize=14, fontweight='bold')
plt.xlabel('Mes del Año')
plt.ylabel('Cantidad de Ventas')
plt.xticks(range(1, 13))

plt.tight_layout()
plt.show()

In [ ]:
import sqlite3

# 1. Creamos la conexión a una base de datos local (se creará el archivo en tu carpeta)
conn = sqlite3.connect('ecommerce_portfolio.db')

# 2. Exportamos el DataFrame limpio a una tabla SQL
# if_exists='replace' asegura que si corrés la celda de nuevo, se actualice sin duplicar
df_master.to_sql('ventas_limpias', conn, if_exists='replace', index=False)

print("¡Datos exportados exitosamente a SQL (ecommerce_portfolio.db)!")

In [ ]:
# Exportamos también los datos limpios a CSV.
# 'ventas.csv' y 'productos.csv' son el "archivo del día" que el pipeline
# automatizado (03_etl_pipeline.py) va a leer y cargar de forma incremental
# a la tabla 'ventas_diarias_automatizadas'. Si no existieran estos archivos,
# 03_etl_pipeline.py genera datos de ejemplo automáticamente como fallback.
df_ventas.to_csv('ventas.csv', index=False, encoding='utf-8')
df_productos.to_csv('productos.csv', index=False, encoding='utf-8')

print("¡Archivos 'ventas.csv' y 'productos.csv' generados para 03_etl_pipeline.py!")

In [ ]:
# Consulta SQL: Análisis de rendimiento de productos
query_top_productos = """
    SELECT 
        nombre AS Producto,
        categoria AS Categoria,
        SUM(cantidad) AS Unidades_Vendidas,
        SUM(ingreso_total) AS Ingreso_Total_USD,
        ROUND(AVG(ingreso_total), 2) AS Ticket_Promedio
    FROM ventas_limpias
    GROUP BY nombre, categoria
    ORDER BY Ingreso_Total_USD DESC
    LIMIT 3;
"""

# Ejecutamos la query y la guardamos en un nuevo DataFrame
df_top_productos_sql = pd.read_sql_query(query_top_productos, conn)

print("--- TOP 3 PRODUCTOS CON MAYOR INGRESO (SQL) ---")
display(df_top_productos_sql)

In [ ]:
query_metodos_pago = """
    SELECT 
        metodo_pago AS Metodo,
        COUNT(id_venta) AS Cantidad_Transacciones,
        SUM(ingreso_total) AS Volumen_Operado_USD,
        ROUND((COUNT(id_venta) * 100.0 / (SELECT COUNT(*) FROM ventas_limpias)), 2) AS Porcentaje_Uso
    FROM ventas_limpias
    GROUP BY metodo_pago
    ORDER BY Volumen_Operado_USD DESC;
"""

df_metodos = pd.read_sql_query(query_metodos_pago, conn)
display(df_metodos)

# Cerramos la conexión a la base de datos por buenas prácticas
conn.close()